# Figure 2 — Differential unproductive splicing across human tissues

Panels **a**, **b** and **e**, drawn as fully vector PDF/SVG with editable text.
The remaining panels of Figure 2 live in sibling notebooks in this directory:

| Panel | Content | Notebook |
|---|---|---|
| **a** | Unproductive junction-read percentage per sample, by tissue | this notebook |
| **b** | Splicing-vs-expression correlation per tissue pair | this notebook |
| **c** | Heatmap of differentially used unproductive events | `Figure2_heatmap.ipynb` (R) |
| **d** | *GABBR1* sashimi plot | `Figure2_prepare_sashimi.ipynb` |
| **e** | *GABBR1* expression across tissues | this notebook |

Cell 3 builds and pickles all plot-ready data; cell 2 reloads it from
`figure_data/` without recomputing. Panels are written to `plots/` as `fig2a`,
`fig2b`, `fig2e`.

*Naming note:* these panels were `fig2A`, `fig2C` and `fig2_boxplots` in the
original `../Fig2.ipynb`. Function names, pickle keys and output files now all
follow the caption letters; pickles written under the old names were moved to
`../old_pickles/`, and figures written under the old names to `../old_figures/`.

In [ ]:
import os
import importlib

from matplotlib import pyplot as plt

import Figure2_plot_helpers
import Figure2_helpers
importlib.reload(Figure2_plot_helpers)
importlib.reload(Figure2_helpers)

from Figure2_plot_helpers import (plot_fig2a, plot_fig2b, plot_gene_boxplots,
                                  boxplot_legend, fig2b_legend, FIG2A_CENTRE_LEGEND)
from Figure2_helpers import run_all, load_plot_data, TEN_TISSUES_CLEAN, BOXPLOT_PALETTE

# Keep text as editable text in the vector output, not outlined shapes.
# svg.fonttype='none'  -> matplotlib emits <text> elements that name the font
#                         instead of converting each glyph to a <path>, so
#                         Inkscape opens them as editable text objects.
# pdf.fonttype=42      -> embeds TrueType rather than Type 3, which keeps text
#                         selectable and editable in PDF editors too.
plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

PLOTS_DIR = 'plots'
os.makedirs(PLOTS_DIR, exist_ok=True)
print('panels will be written to', PLOTS_DIR)


def save_panel(name, dpi=300):
    """Clear every rasterization flag on the current figure, then save it.

    The plot helpers pass rasterized=True to their scatter/strip layers. Walking
    fig.findobj() and calling set_rasterized(False) turns that off without
    editing Figure2_plot_helpers.py, so the PDF and SVG hold real vector paths.
    """
    fig = plt.gcf()
    for artist in fig.findobj():
        if hasattr(artist, 'set_rasterized'):
            try:
                artist.set_rasterized(False)
            except Exception:
                pass
    for ax in fig.axes:
        ax.set_rasterization_zorder(None)

    for ext in ['png', 'pdf', 'svg']:
        fig.savefig(f'{PLOTS_DIR}/{name}.{ext}', dpi=dpi, bbox_inches='tight')
    print('wrote', f'{PLOTS_DIR}/{name}.[png|pdf|svg]')
    return fig

In [ ]:
# Fast path: re-plot straight from the pickles written by run_all().
# Uncomment this cell and skip the next one.

# data = load_plot_data('figure_data')

# fig2a_panels        = data['fig2a_panels']
# fig2a_tissue_names  = data['fig2a_tissue_names']
# fig2b_series        = data['fig2b_series']
# fig2b_source_data   = data['fig2b_source_data']
# fig2e_boxplot_df    = data['fig2e_boxplot_df']
# fig2e_boxplot_stats = data['fig2e_boxplot_stats']

In [ ]:
# Full data pipeline: reads the GTEx leafcutter2 noise tables, the GTEx TPM
# table and the pairwise splicing-vs-expression comparisons, computes the
# Fig. 2e sample sizes and Wilcoxon signed-rank test and the Fig. 2b per-point
# statistics, then pickles every plot-ready variable into figure_data/ and
# writes figure_data/fig2b_source_data.tsv (Supplementary Table 3).
# This is the only heavy cell.

data = run_all('figure_data')

fig2a_panels        = data['fig2a_panels']
fig2a_tissue_names  = data['fig2a_tissue_names']
fig2b_series        = data['fig2b_series']
fig2b_source_data   = data['fig2b_source_data']
fig2e_boxplot_df    = data['fig2e_boxplot_df']
fig2e_boxplot_stats = data['fig2e_boxplot_stats']

### Fig. 2a

Percent of junction reads that are classified as unproductive for each sample,
grouped according to GTEx tissue type.

In [ ]:
# Fig. 2a -- was `fig2A` in ../Fig2.ipynb

plot_fig2a(fig2a_panels, fig2a_tissue_names)
save_panel('fig2a', dpi=300)

### Fig. 2b

Scatter plot showing the two-sided Spearman's correlation between delta PSI and
differential gene expression Z-scores for tissue pairs (x-axis) and the
significance of the correlation (y-axis). Correlations are tested over the set of
genes common between each pair, which varies for each tissue pair. Pairs with
n < 50 are not shown (exact n and test statistics for each pair are shown in
Supplementary Table 3). The dashed line marks unadjusted P = 0.01 and colored
points have an FDR ≤ 10% (Benjamini–Hochberg).

Supplementary Table 3 is written by `run_all` as
`figure_data/fig2b_source_data.tsv`.

In [ ]:
# Fig. 2b -- was `fig2C` in ../Fig2.ipynb
# print(fig2b_legend(fig2b_source_data)) for the generated legend text.

plot_fig2b(fig2b_series)
save_panel('fig2b', dpi=600)

### Fig. 2e

Boxplots of *GABBR1* expression level across GTEx tissues. High expression of
*GABBR1* in brain tissue is associated with high exon inclusion. Each point is
one sample from a distinct GTEx donor, with n given below each box. Boxes depict
the median (red line) and interquartile range (IQR), with whiskers extending to
the most extreme value no greater than 1.5× IQR from the hinge, with all points
overlaid. P values shown are from a two-sided Wilcoxon signed-rank test on
per-donor medians of brain and non-brain tissues.

The caption quotes n = 340 paired donors — confirm against
`fig2e_boxplot_stats['wilcoxon']['n_pairs']` after running cell 3.

In [ ]:
# Fig. 2e -- was `fig2_boxplots` in ../Fig2.ipynb
# print(boxplot_legend(fig2e_boxplot_stats)) for the generated legend text.
# fig2e_boxplot_stats['wilcoxon']['n_pairs'] is the paired-donor n in the caption.

plot_gene_boxplots(fig2e_boxplot_df, 'GABBR1', TEN_TISSUES_CLEAN, BOXPLOT_PALETTE,
                   stats=fig2e_boxplot_stats)
save_panel('fig2e', dpi=300)